# Leakage-Safe Feature Engineering

This notebook explains and inspects the reproducible feature master. It does not train a model, choose a split, or alter the canonical dataset.

In [ ]:
from pathlib import Path
import math
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data' / 'processed').exists():
    ROOT = ROOT.parent
FEATURE_PATH = ROOT / 'data' / 'processed' / 'eia_ciso_hourly_features.csv'
TABLE_DIR = ROOT / 'results' / 'features' / 'tables'
features = pd.read_csv(FEATURE_PATH, parse_dates=['period'])
print(f'Feature master: {len(features):,} rows x {len(features.columns)} columns')
print(f"Timeline: {features['period'].min():%Y-%m-%dT%H} through {features['period'].max():%Y-%m-%dT%H} UTC")
print(f"Available targets: {features['target_available'].sum():,}")

## Why shifting prevents leakage

A model predicting demand at hour `t` must behave as if it is standing immediately before `t`. `shift(1)` aligns the demand reported at `t-1` with row `t`. A rolling window is therefore calculated from a shifted series: a 24-hour window at `t` covers `t-24` through `t-1`. Without the shift, demand at `t` would enter its own predictor and make evaluation unrealistically easy.

In [ ]:
example_columns = ['period', 'demand_mwh', 'demand_lag_1h', 'demand_lag_24h', 'demand_rolling_24h_mean']
example = features.loc[24:28, example_columns]
print(example.to_string(index=False))
assert features.loc[24, 'demand_lag_1h'] == features.loc[23, 'demand_mwh']
assert features.loc[24, 'demand_rolling_24h_mean'] == features.loc[:23, 'demand_mwh'].mean()

## Cyclical calendar features

Hour 23 and hour 0 are neighbours, even though their ordinary numeric values look far apart. Sine and cosine place hours around a circle so midnight wraps smoothly. Day of week uses the same idea with a seven-day period. Day of year uses 365 days in an ordinary year and 366 in a leap year, representing yearly repetition without fitting an encoder to future data. Calendar fields are safe because they are known before the forecast is made.

In [ ]:
cycle_example = features.loc[:23, ['hour_utc', 'hour_sin', 'hour_cos']].iloc[[0, 6, 12, 18, 23]]
print(cycle_example.to_string(index=False))
expected_hour_sin = features['hour_utc'].map(lambda hour: math.sin(2 * math.pi * hour / 24))
assert (features['hour_sin'] - expected_hour_sin).abs().max() < 1e-12

## Why same-hour renewables are unsafe

The reported solar and wind measurements for `t` are actual outcomes at `t`; they are not guaranteed to exist just before demand at `t` is forecast. The declared predictor groups therefore exclude current solar, wind, combined generation, renewable share, residual demand, and the current negative-solar flag. The optional enhanced group uses only lagged renewable observations. Negative solar reports are preserved rather than clipped.

In [ ]:
group_table = pd.read_csv(TABLE_DIR / 'feature_groups.csv')
group_counts = group_table[['calendar_only', 'autoregressive_demand', 'renewable_history_enhanced']].sum().astype(int)
print('Cumulative predictor counts:')
print(group_counts.to_string())
unsafe_predictors = group_table.loc[group_table['safe_at_forecast_time'].eq(False) & group_table['renewable_history_enhanced'].eq(True)]
assert unsafe_predictors.empty

## Missing values propagate by design

A missing source demand remains missing when it reaches a lag. It also makes a full rolling window unavailable for every later row whose history includes that gap. Early rows have the same honest limitation because enough history does not yet exist. Availability flags describe this condition; they never delete or fill rows.

In [ ]:
null_counts = pd.read_csv(TABLE_DIR / 'engineered_feature_null_counts.csv')
history_nulls = null_counts[null_counts['feature'].str.contains('lag|rolling', regex=True)]
print(history_nulls.head(12).to_string(index=False))
assert features['target_demand_mwh'].isna().sum() == 5
assert features['demand_lag_168h'].iloc[:168].isna().all()

## Three cumulative modelling choices

Calendar-only is the smallest group. Autoregressive demand adds past demand lags and past-only rolling summaries. Renewable-history enhanced adds past solar, wind, combined generation, and lagged negative-solar flags. The table below reports availability before any split only for auditing. In later modelling, chronological boundaries must be defined first, and target/feature filtering must happen inside each period.

In [ ]:
eligibility = pd.read_csv(TABLE_DIR / 'feature_group_eligibility.csv')
print(eligibility.to_string(index=False))
assert eligibility['total_rows'].eq(len(features)).all()
print('No model is trained or evaluated in this notebook.')